# TransOrg AgentIQ Datathon — Gate 2: Data Engineering & Rescue Walkthrough
**Track 1: FinTech & BFSI (UPI Fraud & Merchant Analytics)**

---
### 📌 Evaluation Rubric Coverage
1. **Handling Missing Values & Duplicates (10 pts)**: Non-lossy imputation and domain-aware deduplication (Zero lazy row drops).
2. **Standardization (10 pts)**: Regex PAN/Aadhaar validation, currency cleaning (₹, Rs., INR, multiplier suffixes), epoch/multi-format timestamp parsing, and 82->9 merchant category taxonomy harmonization.
3. **Reproducibility (10 pts)**: Self-contained, executable top-to-bottom pipeline creating SQLite relational store `upi_fraud_analytics.db`.
4. **Bonus Commentary (10 pts)**: Granular explanations of data rescue architecture, edge cases, and relational constraints.


## Step 1: Environment Setup & Raw Data Loading
Load all 4 raw datasets: `track1_upi_transactions.csv`, `track1_kyc_records.csv`, `track1_merchants_master.csv`, and `track1_chargebacks.json`.

In [ ]:
import json
import os
import re
from datetime import datetime
import pandas as pd
import numpy as np

from src.normalizers import normalize_user_id, normalize_merchant_id, normalize_mcc
from src.cleaner import clean_transactions, clean_kyc, clean_merchants, clean_chargebacks
from src.database import init_database, get_db_connection
from src.metrics import compute_executive_kpis, compute_merchant_risk_scores, compute_customer_risk_scores

# Load Raw Datasets
df_tx_raw = pd.read_csv('track1_upi_transactions.csv')
df_kyc_raw = pd.read_csv('track1_kyc_records.csv')
df_mch_raw = pd.read_csv('track1_merchants_master.csv')
with open('track1_chargebacks.json', 'r', encoding='utf-8') as f:
    cb_raw_data = json.load(f)

print(f'Raw Transactions Rows : {len(df_tx_raw):,}')
print(f'Raw KYC Records Rows  : {len(df_kyc_raw):,}')
print(f'Raw Merchants Rows    : {len(df_mch_raw):,}')
print(f'Raw Chargebacks Rows  : {len(cb_raw_data):,}')


## Step 2: Canonical ID Normalization Engine
### Decision Rationale
- Raw data contains inconsistent casing (`usr45826` vs `USR45826`), delimiters (`USR-54113`, `usr_12345`), whitespace (`  USR  19283 `), and integer casts (`87810`).
- The canonical normalizer parses numeric digits and rebuilds uniform uppercase prefixes (`USR` and `MCH`), repairing cross-table relationships without dropping unlinked foreign keys.

In [ ]:
sample_users = ['USR45826', 'usr97580', 'USR 45454', 'USR-54113', 'usr_12345', '45454', 87810, '  USR  19283 ']
sample_merchants = ['mch2849', 'MCH4314', '3835', 'MCH-1986', 'mch_9999', 7045, ' MCH 5031 ']

print('--- Canonical ID Demonstration ---')
for u in sample_users:
    print(f'Raw User: {repr(u):<20} -> Canonical: {normalize_user_id(u)}')

print('')
for m in sample_merchants:
    print(f'Raw Merchant: {repr(m):<20} -> Canonical: {normalize_merchant_id(m)}')


## Step 3: Non-Lossy Data Cleaning & Feature Engineering
### Decision Rationale
- **Transactions**: Clean multi-currency prefixes (`₹`, `Rs.`, `INR`), parse Unix epoch & ISO timestamps into structured date/hour/day dimensions, normalize status (`SUCCESS`, `FAILED`, `PENDING`), and deduplicate transaction IDs.
- **Customers (KYC)**: Regex PAN validation (`^[A-Z]{5}[0-9]{4}[A-Z]{1}$`), 12-digit Aadhaar sanitization, monthly income conversion, city alias harmonization (`CALCUTTA` -> `Kolkata`, `BOMBAY` -> `Mumbai`), and DOB/Age calculation.
- **Merchants**: Fix OCR noise in names (e.g. `Gh0sh` -> `Ghosh`), standardize 82 messy category strings into 9 statutory categories, handle missing settlement accounts gracefully, and normalize active status.
- **Chargebacks**: Link disputes to transaction & merchant IDs, compute statutory reporting delays, and bucket reason codes into canonical classifications (`FRAUD_ATO`, `NON_DELIVERY`, `DUPLICATE_DEBIT`, `WRONG_AMOUNT`, `SERVICE_ISSUE`).

In [ ]:
# Execute Cleaners
df_tx, tx_stats = clean_transactions(df_tx_raw)
df_kyc, kyc_stats = clean_kyc(df_kyc_raw)
df_mch, mch_stats = clean_merchants(df_mch_raw)
df_cb, cb_stats = clean_chargebacks(cb_raw_data)

print('=== DATA RESCUE EXECUTION AUDIT ===')
print(f'Transactions Cleaned : {len(df_tx):,} records (Currency fixed: {tx_stats["currency_formats_fixed"]:,}, Epoch dates: {tx_stats["epoch_timestamps_fixed"]:,})')
print(f'KYC Customers Cleaned: {len(df_kyc):,} unique users (Invalid PANs flagged: {kyc_stats["invalid_pan_count"]:,})')
print(f'Merchants Cleaned    : {len(df_mch):,} unique entities (Active: {mch_stats["active_merchant_count"]:,}, Inactive: {mch_stats["inactive_merchant_count"]:,})')
print(f'Disputes Cleaned     : {len(df_cb):,} chargebacks (Total Disputed: Rs. {cb_stats["disputed_amount_total"]:,.2f})')


## Step 4: Relational Schema Creation (SQLite & Indexed Store)
Ingests the 4 cleaned tables into `upi_fraud_analytics.db` with primary keys, foreign keys, and analytical views.

In [ ]:
db_conn = init_database(df_tx, df_kyc, df_mch, df_cb, db_path='upi_fraud_analytics.db')
cursor = db_conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print('Database Tables Initialized:', [t[0] for t in tables])

for t in ['transactions', 'customers', 'merchants', 'chargebacks']:
    cursor.execute(f'SELECT COUNT(*) FROM {t};')
    cnt = cursor.fetchone()[0]
    print(f'  • Table "{t}": {cnt:,} rows')


## Step 5: Core Business Metrics & Multi-Dimensional Risk Models
Computes Executive KPIs, Isolation Forest Anomaly Spikes, and Customer Risk Scores.

In [ ]:
kpis = compute_executive_kpis(df_tx, df_kyc, df_mch, df_cb)
df_mch_risk = compute_merchant_risk_scores(df_mch, df_tx, df_cb)
df_cust_risk = compute_customer_risk_scores(df_kyc, df_tx, df_cb)

print('=== EXECUTIVE DASHBOARD METRICS ===')
print(f'Total Processed Volume : Rs. {kpis["total_amount"]:,.2f}')
print(f'Payment Success Rate   : {kpis["success_rate"]}%')
print(f'Payment Failure Rate   : {kpis["failed_rate"]}%')
print(f'Dispute-to-Txn Ratio   : {kpis["chargeback_to_txn_ratio"]}%')
print(f'KYC Verification Rate  : {kpis["kyc_completion_rate"]}%')
print(f'High Risk Merchants    : {(df_mch_risk["risk_score"] > 60).sum():,} accounts')
print(f'High Risk Customers    : {(df_cust_risk["risk_score"] > 60).sum():,} entities')

# Preview Top 5 High-Risk Merchants
df_mch_risk.sort_values(by='risk_score', ascending=False)[['merchant_id', 'merchant_name_clean', 'cb_ratio_pct', 'risk_score', 'risk_segment']].head(5)
